# 🚩 शिव एआई (Shiv AI) - ३२ चाइल्ड बॉट्स मास्टर ट्रेनिंग सेंटर
**मालिक:** श्री राम नाग
इसमें प्रोग्रेस बार, गूगल ड्राइव बैकअप और री-ट्रेनिंग की सुविधा है।

In [ ]:
# @title ⚙️ सेटअप और टर्बो लाइब्रेरीज़ (३२ बॉट्स सिंक)
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install -q wikipedia requests datasets
print("✅ शिव एआई ३२ बॉट्स के लिए तैयार है!")

In [ ]:
# @title 📂 री-ट्रेनिंग फॉर्म (ड्राइव से पुराना ज्ञान लोड करें)
from google.colab import drive
import os
from unsloth import FastLanguageModel

drive.mount('/content/drive')
LOAD_FROM_DRIVE = False #@param {type:"boolean"}
MODEL_PATH = "/content/drive/MyDrive/ShivAI_Final_Model" #@param {type:"string"}

if LOAD_FROM_DRIVE and os.path.exists(MODEL_PATH):
    print(f"🔄 ३२ बॉट्स का पुराना ज्ञान लोड हो रहा है...")
    model, tokenizer = FastLanguageModel.from_pretrained(model_name = MODEL_PATH, max_seq_length = 2048, load_in_4bit = True)
else:
    print("🆕 नया मास्टर मॉडल लोड किया जा रहा है...")
    model, tokenizer = FastLanguageModel.from_pretrained(model_name = "HuggingFaceTB/SmolLM2-135M-Instruct", max_seq_length = 2048, load_in_4bit = True)

In [ ]:
# @title 🌐 इंटरनेट से डेटा खींचना (Wiki/GitHub)
import wikipedia
import requests
from datasets import Dataset

wikipedia.set_lang("hi")
topics = ["मशीन लर्निंग", "साइबर सुरक्षा", "आयुर्वेद"]
kb = []
tags = ["[CODE_EXPERT]", "[CYBER_SEC]", "[AYURVEDA]"]

for i, t in enumerate(topics):
    try:
        p = wikipedia.page(t)
        kb.append({"instruction": f"{tags[i]} {t} क्या है?", "response": p.summary[:1000]})
    except: pass

dataset = Dataset.from_list(kb)
print(f"✅ {len(kb)} विषयों का डेटा तैयार है!")

In [ ]:
# @title 🚀 प्रोग्रेस बार के साथ टर्बो ट्रेनिंग
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model, tokenizer = tokenizer, train_dataset = dataset, dataset_text_field = "instruction",
    args = TrainingArguments(
        max_steps = 60, logging_steps = 1, # यह प्रोग्रेस बार दिखाएगा
        output_dir = "outputs", report_to = "none",
    ),
)
trainer.train()

In [ ]:
# @title 📥 ड्राइव में सुरक्षित करें और डाउनलोड करें
save_dir = "/content/drive/MyDrive/ShivAI_Final_Model"
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
os.system(f"zip -r ShivAI_Backup.zip {save_dir}")
from google.colab import files
files.download('ShivAI_Backup.zip')
print("✅ ३२ बॉट्स का ज्ञान सुरक्षित और डाउनलोड के लिए तैयार है!")